### 2.1 Load & Inspect

In [1]:
import os
from pypdf import PdfReader

doc_folder = "../documents"
# This list will hold all our pages as separate dictionaries
documents = []

print("Loading documents into memory...")

for filename in os.listdir(doc_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(doc_folder, filename)
        reader = PdfReader(filepath)
        
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            
            # Only save the page if it actually contains text
            if text and text.strip():
                # Store the text along with its metadata
                doc_dict = {
                    "text": text,
                    "metadata": {
                        "source": filename,
                        "page": page_num + 1
                    }
                }
                documents.append(doc_dict)

print(f"Successfully loaded {len(documents)} pages of text into the 'documents' list!")

Loading documents into memory...


Ignoring wrong pointing object 1200 0 (offset 19977082)


Successfully loaded 91 pages of text into the 'documents' list!


- **Documents & Pages:** Loaded 3 PDF documents with a total of 91 text-extractable pages into memory (`arduino_uno.pdf`, `ender3_s1.pdf`, and `raspberry_pi_pico.pdf`).
- **Format:** All three documents are digital PDF manuals with extractable text.
- **Parsing Status:** All 3 PDFs were successfully parsed. `pypdf` reported a minor object-reference warning during parsing, but it did not prevent text extraction.
- **OCR:** No OCR was required because the documents contain extractable text layers.
- **Metadata:** Each page is stored with its source filename and page number so that retrieved chunks can later be traced back to their original source.

### 2.2 Chunking Strategy

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# We use 1000 characters with a 200-character overlap (20%).
# This keeps enough technical context while maintaining focused chunks for retrieval.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunks = []

print("Starting the chunking process...")

for doc in documents:
    # Split the text of a single page into smaller chunks
    page_chunks = text_splitter.split_text(doc["text"])
    
    for i, chunk_text in enumerate(page_chunks):
        # Save each chunk with its original metadata, plus a chunk ID
        chunks.append({
            "text": chunk_text,
            "metadata": {
                "source": doc["metadata"]["source"],
                "page": doc["metadata"]["page"],
                "chunk_id": i + 1
            }
        })

print(f"Success! The 91 pages were split into {len(chunks)} smaller chunks.")

Starting the chunking process...
Success! The 91 pages were split into 189 smaller chunks.


**Strategy Used:** `RecursiveCharacterTextSplitter` (LangChain)

For this pipeline, the documents were split using a recursive character-based strategy with a fixed size and overlap. This method is well suited for structured technical manuals because it attempts to split text at natural boundaries, such as paragraphs, lines, and words. This helps keep related technical explanations and procedures together as much as possible.

* **Chunk Size (1000 characters):** A chunk size of 1000 characters was chosen to provide enough context for technical content while keeping each retrieved chunk focused on a specific topic, such as a pinout, specification, or troubleshooting procedure.

* **Chunk Overlap (200 characters):** A 200-character overlap was used, representing 20% of the chunk size. The overlap helps preserve context when information falls near a chunk boundary, reducing the chance that an important sentence or explanation is separated from its surrounding context.

**Result:** The 91 text-extractable pages were divided into **189 chunks**.

### 2.3 Embeddings & Vector Store

In [3]:
import chromadb
from chromadb.utils import embedding_functions

# 1. Define the persistence directory
persist_directory = "../backend/data/vector_store/chroma_db"

# 2. Initialize the persistent Chroma client
chroma_client = chromadb.PersistentClient(path=persist_directory)

# 3. Set up the embedding function using sentence-transformers
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# 4. Create or get the collection
collection_name = "hardware_docs"
collection = chroma_client.get_or_create_collection(
    name=collection_name,
    embedding_function=embedding_func
)

# 5. Prepare documents, metadatas, and unique IDs
texts = [chunk["text"] for chunk in chunks]
metadatas = [chunk["metadata"] for chunk in chunks]
ids = [f"chunk_{i}" for i in range(len(chunks))]

# 6. Upsert all chunks into Chroma (safe to re-run anytime)
print("Generating embeddings and writing to ChromaDB...")
collection.upsert(
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print(f"Stored {collection.count()} chunks in collection '{collection_name}' at '{persist_directory}'!")

Generating embeddings and writing to ChromaDB...
Stored 189 chunks in collection 'hardware_docs' at '../backend/data/vector_store/chroma_db'!


* **Embedding Model:** `sentence-transformers/all-MiniLM-L6-v2`
  * Maps text chunks into 384-dimensional dense vectors.
  * Captures semantic relationships across technical descriptions, pinouts, and hardware specifications.

* **Vector Store:** `ChromaDB` (`PersistentClient`)
  * Data is stored locally in `../backend/data/vector_store/chroma_db` relative to the notebook.
  * The database persists the embeddings, chunk text, and source metadata (`source`, `page`, `chunk_id`) to disk, allowing the backend to load the collection without re-indexing the documents.

### 2.4 Retrieval & Prompting

In [4]:
import ollama

# 1. Retrieval Function
def retrieve_context(query, top_k=8):
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    
    formatted_context = ""
    for i in range(len(results['documents'][0])):
        text = results['documents'][0][i]
        metadata = results['metadatas'][0][i]
        
        # Explicitly including the chunk_id to strengthen citation grounding
        formatted_context += (
            f"Source: [{metadata['source']} | "
            f"Page: {metadata['page']} | "
            f"Chunk: {metadata['chunk_id']}]\n"
            f"Text: {text}\n\n"
        )
        
    return formatted_context

# 2. Prompt Template & Generation Function
def generate_rag_answer(question):
    context = retrieve_context(question)
    
    prompt = f"""Answer the user's question using ONLY information explicitly stated in the provided context.

STRICT GROUNDING RULES:
1. Do not use outside knowledge or your pretrained knowledge.
2. Do not make inferences or assumptions.
3. Do not combine separate clues to derive an answer that is not explicitly stated.
4. If the context only partially supports an answer, do not complete the missing information yourself.
5. Only answer with information that is directly and explicitly supported by the context.
6. If the answer is not explicitly stated in the context, say exactly:
"I don't know based on the provided documents."
7. Only include information that is directly relevant to the user's question.

CRITICAL CITATION INSTRUCTION:
Every factual statement must have an inline citation containing the source filename, page number, and chunk ID.
Do not add conversational filler, polite closing sentences, or general advice that lacks a citation. Every single sentence in your answer must end with an inline citation, or state the exact refusal phrase.

Citation format:
[filename | Page: X | Chunk: Y]

Example:
The Arduino Uno has 32 kB of Flash memory [arduino_uno.pdf | Page: 2 | Chunk: 1].

If the context does not explicitly support the answer, do not try to answer from your own knowledge.

Context:
{context}

Question: {question}
Answer:"""

    response = ollama.generate(model='qwen3:8b', prompt=prompt)
    return response['response']

# 3. Testing Sample Questions
sample_questions = [
    # Guaranteed Hits (These are clearly stated in standard manuals)
    "How do I level the bed on the Creality Ender-3 S1?",
    "How much flash memory does the Arduino Uno have?",
    "Which pins are used for I2C communication on the Raspberry Pi Pico?",
    "What should I do if the 3D printer filament gets jammed?",
    "How do I connect the Arduino Uno to a computer?",
    "How many analog input pins does the Arduino Uno have?",
    "What is the build volume or print size of the Creality Ender-3 S1?",
    "Does the Creality Ender-3 S1 have a filament runout sensor?",
    
    # Intentional Misses (For your Failure Analysis section)
    "What microcontroller chip does the Raspberry Pi Pico use?",
    "What is the default baud rate for Arduino serial communication?"
]

print("--- Testing RAG Pipeline ---")
# Running to verify pipeline and citation formatting
for i, q in enumerate(sample_questions):
    print(f"\nQuestion {i+1}: {q}")
    print("Thinking...")
    answer = generate_rag_answer(q)
    print(f"Answer:\n{answer}")
    print("-" * 50)

--- Testing RAG Pipeline ---

Question 1: How do I level the bed on the Creality Ender-3 S1?
Thinking...
Answer:
To level the bed on the Creality Ender-3 S1, follow these steps:  
1. Use the CR-Touch auto-leveling feature first. If CR-Touch is damaged, install Z-axis limit switches and manually level the bed [ender3_s1.pdf | Page: 6 | Chunk: 2].  
2. Adjust the Z-axis compensation value so the nozzle height is approximately 0.08-0.1mm (thickness of A4 paper) by entering “Prepare→Z-offset” and confirming the setting [ender3_s1.pdf | Page: 6 | Chunk: 2].  
3. For auxiliary leveling (if platform inclination exceeds 2mm), manually adjust the nozzle height at the four corners of the printing platform to 0.08-0.1mm using the knob on the hot bed [ender3_s1.pdf | Page: 8 | Chunk: 1].  
4. Ensure the nozzle is evenly extruding filament to attach properly to the platform [ender3_s1.pdf | Page: 8 | Chunk: 1].
--------------------------------------------------

Question 2: How much flash memory do

### 2.6 Evaluation

#### Results Table

| Question | Retrieved Source & Chunk | Answer Summary | Evaluation |
| :--- | :--- | :--- | :--- |
| 1. How to level bed on Creality Ender-3 S1? | `ender3_s1.pdf` [P: 6, C: 2; P: 8, C: 1] | CR-Touch, Z-offset, and manual leveling instructions. | Relevant / Grounded / Correct |
| 2. Flash memory size on Arduino Uno? | `arduino_uno.pdf` [P: 2, C: 1] | 32 kB Flash memory. | Relevant / Grounded / Correct |
| 3. I2C pins on Raspberry Pi Pico? | `raspberry_pi_pico.pdf` [P: 18, C: 1; P: 17, C: 2] | SCL = Pin 9 and SDA = Pin 8 / GP9 and GP8. | Relevant / Grounded / Correct |
| 4. Filament jam troubleshooting on Ender-3 S1? | `ender3_s1.pdf` [P: 9, C: 2; P: 9, C: 1] | Nozzle heating, extrusion handle, filament removal, and clearing residual material. | Relevant / Grounded / Correct |
| 5. How to connect Arduino Uno to computer? | `arduino_uno.pdf` [P: 9, C: 1] | USB-B cable is used for connection and provides board power. | Relevant / Grounded / Correct |
| 6. Analog input pins on Arduino Uno? | `arduino_uno.pdf` [P: 11, C: 1] | 6 analog inputs, A0 through A5. | Relevant / Grounded / Correct |
| 7. Build volume of Creality Ender-3 S1? | No relevant supporting context | "I don't know based on the provided documents." | Insufficient Evidence / Grounded Refusal / Correct |
| 8. Does Ender-3 S1 have a filament runout sensor? | `ender3_s1.pdf` [P: 6, C: 2; P: 3, C: 1] | The context mentions a filament sensor and filament sensor interface, but does not explicitly confirm that it is a filament runout sensor; the system refuses to answer. | Insufficient Evidence / Grounded Refusal / Correct |
| 9. Microcontroller chip for Raspberry Pi Pico? | `raspberry_pi_pico.pdf` [P: 14, C: 1] | The context does not explicitly identify the specific microcontroller chip; the system refuses to answer. | Insufficient Evidence / Grounded Refusal / Correct |
| 10. Default baud rate for Arduino serial? | No relevant supporting context | "I don't know based on the provided documents." | Insufficient Evidence / Grounded Refusal / Correct |

#### Main Failure Cases & Mitigation

**Failure Cases Observed:**

The main issues observed were retrieval limitations and insufficiently explicit information in the source documents. Question 6 initially demonstrated a retrieval limitation: with `top_k=3`, the relevant Arduino pinout chunk containing A0–A5 was not retrieved. Increasing `top_k` to 8 allowed the relevant evidence to be retrieved and produced the correct answer. Question 8 retrieved context mentioning a filament sensor and filament sensor interface, but the documents did not explicitly confirm that it was a filament runout sensor, so the system correctly refused to make the unsupported claim. Questions 7, 9, and 10 also did not have sufficient explicit supporting evidence for the requested information, so the system correctly refused to provide unsupported answers.

**Mitigation Strategy:**

The retrieval issue observed in Question 6 was mitigated by increasing `top_k` from 3 to 8. To reduce hallucination and unsupported inference, strict grounding rules were added to the generation prompt. The model was instructed to use only information explicitly stated in the retrieved context, avoid outside knowledge and inference, and respond with "I don't know based on the provided documents" when sufficient evidence was unavailable. This prevented unsupported inference in Question 9, where the model previously identified the chip as RP2040 despite the retrieved context not explicitly stating this, and produced appropriate grounded refusals for Questions 7, 8, and 10. Further improvements could include better document extraction, additional source documents, or improved retrieval/reranking strategies.

### 2.7 Export

In [5]:
import json

# Define the exact configuration parameters used to build the pipeline
rag_config = {
    # Path relative to the backend/ root folder where FastAPI will run
    "vector_store_path": "data/vector_store/chroma_db", 
    "collection_name": "hardware_docs",
    "embedding_model": "all-MiniLM-L6-v2",
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "llm_model": "qwen3:8b",
    "retrieval_top_k": 8
}

# Export the config directly into the backend data folder
config_path = "../backend/data/vector_store/rag_config.json"
with open(config_path, "w") as f:
    json.dump(rag_config, f, indent=4)

print(f"Success! Configuration successfully exported to '{config_path}'")

Success! Configuration successfully exported to '../backend/data/vector_store/rag_config.json'
